# Guidelines for data treatment with SCARLET

This notebook, present how to treat data with SCARLET python API

## 1. Workflow initialization: 

In [1]:
from scarlet.workflow.context import initialize_workflow_context_from_raw_directory, WorkflowContext

RAW_DIR = "/home/achennev/python/scarlet/data/SAM/rawdata/" # change this to the path where your raw data is located
OUTPUT_DIR = "/home/achennev/python/scarlet/data/SAM/processed/" # change this to the path where you want to store the output data
INSTRUMENT_NAME = "SAM" # instrument name, used to find the correct configuration files

RAW_DIR, OUTPUT_DIR


('/home/achennev/python/scarlet/data/SAM/rawdata/',
 '/home/achennev/python/scarlet/data/SAM/processed/')

Initialize the work flow context using the `initialize_workflow_context_from_raw_directory` function. This function perform several task : 

 - Read all the data files present in the raw data driectory, convert them into a nexus file with a generic file architecure and save them into the output_directory. 

 - Make a list of instrument configuration used during the experiment (collimation, detector distance, wavelength)

 - Detect if data file corresponds to "empty_cell", "dark", "water" or "empty_beam" measurements based on the sample name.


In [2]:
w = initialize_workflow_context_from_raw_directory(
    RAW_DIR,
    instrument_name=INSTRUMENT_NAME,
    transmission_strategy="opaque_beamstop",
    output_dir=OUTPUT_DIR,
    overwrite=True,
)
# If orverwrite is set to True, the output directory will be cleared before running the workflow. If set to False, the workflow will not run if the output directory already exists.

## 2. Save workflow and filter files
Sometimes, the raw_data folder contains many files that are not necessary for the data treatement and complexify the workflow operation. The 'WorkflowContext ' shall only contains runs intended to be treated and references (empty_cell,...).

For this prupose, a manual filtering or the edition of a csv file is possible.

First save the current state of the workflow in a csv file using the command:

In [3]:
w.write_runs_table_csv(OUTPUT_DIR + 'runs_table.csv', overwrite=True)

PosixPath('/home/achennev/python/scarlet/data/SAM/processed/runs_table.csv')

After editing the csv file, you can run the workflow with the following code in order to load the filtered csv files. The runs can be displayed using the `w.runs_table()` command.

In [4]:
w.update_from_runs_table_csv(OUTPUT_DIR + "runs_table_filtered.csv")
w.runs_table()


sample_name,config_id,mode,entity,thickness,transmission,file_path
EB,config_1,transmission,empty_beam,,,032221.nxs
EC,config_1,transmission,empty_cell,,,032222.nxs
BB,config_1,transmission,dark,,,032223.nxs
H2O,config_1,transmission,water,1,,032224.nxs
RuPil_BuCl_50gL,config_1,transmission,sample,2,,032225.nxs
RuPil_BuCl_20gL,config_1,transmission,sample,2,,032226.nxs
RuPil_BuBr_50gL,config_1,transmission,sample,2,,032227.nxs
RuPil_BuBr_20gL,config_1,transmission,sample,2,,032228.nxs
RuPil_ProBr_50gL,config_1,transmission,sample,2,,032229.nxs
RuPil_ProBr_20gL,config_1,transmission,sample,2,,032230.nxs


The entity column serves as a sample description. It can be set to :
- sample : normal sample
- water: water file allowing to compute flatfield
- empty_cell : empty_cell for substraction
- dark: B4C, Cd file for substraction
- empty_beam : note that a transmission empty_beam is required to compute the transmissions. 

The mode column can also be changed in case of miss recognition. 

The config_id column is just a simple identification string per confiruration used. The configurations parameters can be displayed using the `WorkflowContext.configurations_table()` method

In [5]:
w.configurations_table()

config_id,wavelength,sample_detector_distance,collimation_distance,last_aperture_to_sample_distance,aperture1,aperture2,notes
config_1,5.19612 A,0.84999 m,2.5 m,0.1 m,slit x=0.030992 m y=0.031002 m,pinhole d=0.007 m,
config_2,5.19772 A,2.25016 m,5 m,0.1 m,slit x=0.030992 m y=0.031002 m,pinhole d=0.007 m,
config_3,5.19644 A,6.80001 m,9 m,0.1 m,slit x=0.030992 m y=0.031002 m,pinhole d=0.007 m,
config_4,14.9069 A,6.80002 m,9 m,0.1 m,slit x=0.030992 m y=0.031002 m,pinhole d=0.007 m,


The `WorkflowContext` can be saved a a nexus file for futur use using the following commands:

In [6]:

w.save("workflow.nxs", overwrite=True)
# load the workflow context from the saved file
w = WorkflowContext.load("workflow.nxs")

## 3. Transmission computation

The computation of the detector is performed using the detector named `detector0` in the nexus files refering to the central detector. During the initialization of the `WorkflowContext`, the ROI for transmission measurements is automatically detected.

The ROIs per config can be retrieved and modified using the following methods:

In [7]:
# read roi
print(w.get_roi('config_1'))
# set roi
# w.set_roi('config_10', [61, 65, 62, 64])  # Example ROI coordinates
# w.set_roi('config_9', [60, 64, 61, 63])  # Example ROI coordinates

(123, 133, 62, 68)


The computation of the transmission is perfomd using the `WorkflowContext.compute_transmissions()` method:

In [8]:
w.compute_transmissions()
# save transmission in csv file once computed
w.write_runs_table_csv("OUTPUT_DIR + 'runs_table_with_tr.csv'",overwrite=True)
w.runs_table()

sample_name,config_id,mode,entity,thickness,transmission,file_path
EB,config_1,transmission,empty_beam,,,032221.nxs
EC,config_1,transmission,empty_cell,,0.949452,032222.nxs
BB,config_1,transmission,dark,,,032223.nxs
H2O,config_1,transmission,water,1,0.515061,032224.nxs
RuPil_BuCl_50gL,config_1,transmission,sample,2,0.637874,032225.nxs
RuPil_BuCl_20gL,config_1,transmission,sample,2,0.651094,032226.nxs
RuPil_BuBr_50gL,config_1,transmission,sample,2,0.638728,032227.nxs
RuPil_BuBr_20gL,config_1,transmission,sample,2,0.651042,032228.nxs
RuPil_ProBr_50gL,config_1,transmission,sample,2,0.640647,032229.nxs
RuPil_ProBr_20gL,config_1,transmission,sample,2,0.652503,032230.nxs


## 4. Masking


The edition of masks can be done using the `scarlet viewer` GUI. To run this graphical interface just run `scarlet viewer` command in a terminal with the appropriate virtual environnment activated. The mask files shall be saved in the output directory in order to be found by the workflow.

Each mask file stores the configuration over which they have been drawn. The `WorkflowContext` can automatically adress them to a specc

In [13]:
w.attach_mask_bundles_from_output_dir()
# display the mask files for a specific configuration
w.mask_files

{'config_1': PosixPath('/home/achennev/python/scarlet/data/SAM/processed/mask_config1.nxs'),
 'config_3': PosixPath('/home/achennev/python/scarlet/data/SAM/processed/mask_config3.nxs'),
 'config_4': PosixPath('/home/achennev/python/scarlet/data/SAM/processed/mask_config4.nxs'),
 'config_2': PosixPath('/home/achennev/python/scarlet/data/SAM/processed/masks_config2.nxs')}

Manual attibution of a mask file can be done using the following command:

In [14]:
# w.set_mask_file(config_id='config_10', file_path='/path/to/mask_file')

## 5. Flatfield / water normalization

The `water` entity in the run table can be used to generate flatfields files for normalization by water.
To build a flatfield file for a specific configuration use the following command:

In [15]:
w.build_water_flatfield(config_id="config_1")
w.build_water_flatfield(config_id="config_2")
w.build_water_flatfield(config_id="config_3")


PosixPath('/home/achennev/python/scarlet/data/SAM/processed/flatfield_config_3.nxs')

The flatflied file is saved in the output directory. It can also be set manually with:

In [16]:
# set flatfield file for a specific configuration
# w.set_flatfield(config_id="config_10", file_path="flat_field_path")
# get the flatfield file for a specific configuration
w.get_flatfield(config_id="config_9")

If no water file is available at a given configuration we can declare to use the flatfield of one config to another one using the following command.

In [17]:
w.set_flatfield_source(config_id="config_4", source_config_id="config_2")
# w.get_flatfield(config_id="config_9")

'config_2'

## 6. Reduction pipeline

Once the workflow is set, the data reduction can be performed using a `Pipeline`. The default option perform the following reduction steps:
- Subtract dark and empty_cell
- Division by water flatfield               
- Azimutal averaging
- save processed data in the nexus files in the `processed` entry
- save the processed data as a .txt file for each configuration and each detector.

The reducrion can be perform for a given sample at a given configuration using:

In [18]:
from scarlet.workflow.pipeline import ReductionPipeline, ReductionState
pipeline = ReductionPipeline().default()
state = pipeline.run_for_sample(workflow=w, sample_name="RuPil_BuCl_50gL", config_id="config_1")

Or for the all workflow using:

In [19]:
pipeline.run_all(w)

The pipleline reduction steps can be enriched with other reduction scheme. If you have any suggestion, please contact me.

## 6. Concatenation of configurations

In [ ]:
from scarlet.workflow.pipeline import StichingPipeline
stitch = StichingPipeline(w)
stitch.run_all(scale_on="config_2")

In [ ]:
w.beam_centers

{'config_1': {0: (64.05600379521202, 127.37534577241983)},
 'config_2': {0: (64.14683457824741, 127.54541264346877)},
 'config_3': {0: (64.51615344177192, 126.74968447711399)},
 'config_4': {0: (62.95655366861189, 126.5736607435578)}}